# 05c - HayFlow-Hines causal isolation

This notebook does not authorize or run the full curriculum. It identifies the exact 05b peak failure, decomposes the H2 boundary update, performs nested 1/8/32/76 micro-overfits, and isolates the train counterfactual pair.

## 1. Coherent checkout and runtime

In [ ]:
import os, subprocess, sys
from pathlib import Path
WORKSPACE = Path('/kaggle/working/hayflow_workspace')
ELM_REPO = WORKSPACE / 'elmneuron'
if not ELM_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'pandas', 'pyarrow', 'pyyaml'], check=True)
sys.path.insert(0, str(ELM_REPO))
REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Revision:', REVISION)

In [ ]:
import h5py, json, numpy as np, pandas as pd, pyarrow, torch, yaml
assert torch.cuda.is_available(), 'Attiva una GPU Kaggle prima di eseguire 05c.'
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0)})

## 2. Inputs
Required: complete targeted v1.1 base, BAP top-up v3, and the exact `hayflow_hines_canary_v2.zip` downloaded from 05b.

In [ ]:
import shutil, zipfile
INPUT_ROOT = Path('/kaggle/input')
def extract_zip_safely(source, destination):
    source, destination = Path(source), Path(destination)
    marker = destination / '.source_size'
    stamp = str(source.stat().st_size)
    if marker.is_file() and marker.read_text().strip() == stamp: return destination
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True)
    root = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert target == root or root in target.parents, member.filename
        archive.extractall(destination)
    marker.write_text(stamp)
    return destination

topup_override = os.environ.get('HAYFLOW_TOPUP_V3')
topup_candidates = [Path(topup_override).expanduser()] if topup_override else []
topup_candidates.extend(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))
topup_candidates.extend(path.parent for path in INPUT_ROOT.rglob('composite_dataset_manifest.json'))
TOPUP_SOURCE = next((p.resolve() for p in topup_candidates if p.exists()), None)
assert TOPUP_SOURCE is not None, 'Top-up BAP v3 non trovato.'
TOPUP_ROOT = extract_zip_safely(TOPUP_SOURCE, '/kaggle/working/hayflow05c_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates = list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'))
assert len(manifest_candidates) == 1, manifest_candidates
COMPOSITE_MANIFEST = manifest_candidates[0]

base_override = os.environ.get('HAYFLOW_BASE_DATASET')
base_candidates = [Path(base_override).expanduser()] if base_override else []
base_candidates.extend(path.parent for path in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(path).lower() and 'topup' not in str(path).lower())
base_candidates.extend(path for path in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(path).lower())
BASE_SOURCE = next((p.resolve() for p in base_candidates if p.exists()), None)
assert BASE_SOURCE is not None, 'Dataset base targeted v1.1 non trovato.'

checkpoint_override = os.environ.get('HAYFLOW_05B_ARTIFACT')
checkpoint_candidates = [Path(checkpoint_override).expanduser()] if checkpoint_override else []
checkpoint_candidates.extend(INPUT_ROOT.rglob('hayflow_hines_canary_v2.zip'))
CHECKPOINT_ARCHIVE = next((p.resolve() for p in checkpoint_candidates if p.exists()), None)
assert CHECKPOINT_ARCHIVE is not None, 'Artefatto hayflow_hines_canary_v2.zip non trovato.'
print({'manifest': str(COMPOSITE_MANIFEST), 'base': str(BASE_SOURCE), 'checkpoint': str(CHECKPOINT_ARCHIVE)})

## 3. Cryptographic dataset preflight

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started, hash_last = {}, {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now)
    percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9)
        eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 05c][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True)
        hash_last[name] = percent
bundle = prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST, base_source=BASE_SOURCE, progress=hash_progress)
bundle_summary = {'valid': bool(bundle.manifest['valid']), 'fingerprint': bundle.fingerprint, 'transition_count': bundle.transition_count, 'physical_merge_performed': bool(bundle.manifest['physical_merge_performed'])}
display(bundle_summary)
assert bundle_summary['valid'] and bundle_summary['transition_count'] == 29880
assert not bundle_summary['physical_merge_performed']

In [ ]:
from src.hayflow_model import HinesCausalIsolationExperiment, HinesIsolationConfig, HinesPrototypeExperimentConfig
raw = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_causal_isolation.yml').read_text())
model_config = HinesPrototypeExperimentConfig.from_mapping(raw['model_experiment'])
isolation_config = HinesIsolationConfig.from_mapping(raw['isolation'])
OUTPUT_DIR = Path('/kaggle/working/artifacts/hayflow_hines_causal_isolation')
session = HinesCausalIsolationExperiment(bundle, OUTPUT_DIR, model_config, isolation_config, CHECKPOINT_ARCHIVE)
prepare_report = session.prepare_isolation()
display(prepare_report)
assert prepare_report['training_contract_blockers'] == []

## 4. Independent Hines checks and 05b checkpoint forensics

In [ ]:
hines_report = session.run_hines_layer_tests()
assert hines_report['valid']
checkpoint_report = session.diagnose_checkpoint()
display(checkpoint_report)

## 5. Nested progressive micro-overfit
This is the main cell. Fresh H2 models compare the original timed/masked event boundary with a direct per-segment boundary residual over nested 1/8/32/76 sets. Auxiliary biological targets are excluded.

In [ ]:
progressive_report = session.run_progressive_isolation()
display(progressive_report)

## 6. Counterfactual branch-pair isolation

In [ ]:
branch_report = session.run_branch_isolation()
display(branch_report)

## 7. Diagnostic decision and artifact contract

In [ ]:
final_report = session.finalize_isolation(checkpoint_report, progressive_report, branch_report)
display(final_report)
assert final_report['valid']
assert not final_report['full_training_authorized']
required = ['final_report.json', 'artifact_index.json', 'checkpoint_forensics.json', 'checkpoint_transition_errors.parquet', 'worst_transition_segment_decomposition.parquet', 'worst_transition_events.json', 'progressive_overfit_metrics.parquet', 'branch_isolation_metrics.parquet']
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
assert not missing, missing
print({'diagnosis': final_report['diagnosis'], 'output_dir': str(OUTPUT_DIR), 'missing': missing})

## 8. Browser download with checkpoints

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript, display
zip_base = Path('/kaggle/working/hayflow_hines_causal_isolation')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
filename = zip_path.name
display(Javascript(f'''
const binary = atob('{encoded}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
'''))
print('Download avviato:', filename, f'({zip_path.stat().st_size / 2**20:.1f} MiB)')